Lanette Tyler   
Homework 10   
ST554   
Spring 2026   

# Part 1: Create Streaming Data Using `rate`

In [1]:
#create spark session
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/21 21:03:46 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/21 21:03:47 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [2]:
#create streaming data frame object from rate
rateDF = spark.readStream.format("rate").load()

In [3]:
#view rate data frame
rateDF

DataFrame[timestamp: timestamp, value: bigint]

In [4]:
#add transformations to rate data frame

#import functions
from pyspark.sql.functions import col, sqrt

#modify data frame
modRateDF = rateDF.withColumn("sqrt_value", sqrt(col("value")))
modRateDF = modRateDF.withColumn("mod_4", col("value") % 4)

#view modified data frame object
modRateDF

DataFrame[timestamp: timestamp, value: bigint, sqrt_value: double, mod_4: bigint]

In [5]:
#write streaming data as a table to memory, start it
writeTable = modRateDF.writeStream.format("memory").queryName("my_table").start()

26/04/21 21:05:07 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-e324ac74-a002-4adb-af30-3c2344e1362c. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/21 21:05:07 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


In [6]:
#stop data stream
writeTable.stop()

26/04/21 21:05:44 WARN DAGScheduler: Failed to cancel job group ac6d5371-ffdf-434d-b1a9-aafc062e8dcb. Cannot find active jobs for it.
26/04/21 21:05:44 WARN DAGScheduler: Failed to cancel job group ac6d5371-ffdf-434d-b1a9-aafc062e8dcb. Cannot find active jobs for it.


In [7]:
#view table
spark.sql("SELECT * FROM my_table").show()

+--------------------+-----+------------------+-----+
|           timestamp|value|        sqrt_value|mod_4|
+--------------------+-----+------------------+-----+
|2026-04-21 21:05:...|    0|               0.0|    0|
|2026-04-21 21:05:...|    1|               1.0|    1|
|2026-04-21 21:05:...|    2|1.4142135623730951|    2|
|2026-04-21 21:05:...|    3|1.7320508075688772|    3|
|2026-04-21 21:05:...|    4|               2.0|    0|
|2026-04-21 21:05:...|    5|  2.23606797749979|    1|
|2026-04-21 21:05:...|    6| 2.449489742783178|    2|
|2026-04-21 21:05:...|    7|2.6457513110645907|    3|
|2026-04-21 21:05:...|    8|2.8284271247461903|    0|
|2026-04-21 21:05:...|    9|               3.0|    1|
|2026-04-21 21:05:...|   10|3.1622776601683795|    2|
|2026-04-21 21:05:...|   11|   3.3166247903554|    3|
|2026-04-21 21:05:...|   12|3.4641016151377544|    0|
|2026-04-21 21:05:...|   13| 3.605551275463989|    1|
|2026-04-21 21:05:...|   14|3.7416573867739413|    2|
|2026-04-21 21:05:...|   15|

# Part 2: Use Data from a CSV with a Pipeline

In [8]:
#read in bike details for fit as a spark SQL data frame
fitDetails = spark.read.load("bikeDetails_for_fit.csv",
                     format = "csv",
                     sep = ",",
                     inferSchema = "true",
                     header = "true")
fitDetails.show(5)
fitDetails

+--------------------+-------------+----+-----------+---------+---------+-----------------+
|                name|selling_price|year|seller_type|    owner|km_driven|ex_showroom_price|
+--------------------+-------------+----+-----------+---------+---------+-----------------+
|Royal Enfield Cla...|       175000|2019| Individual|1st owner|      350|             NULL|
|           Honda Dio|        45000|2017| Individual|1st owner|     5650|             NULL|
|Royal Enfield Cla...|       150000|2018| Individual|1st owner|    12000|           148114|
|Yamaha Fazer FI V...|        65000|2015| Individual|1st owner|    23000|            89643|
|Yamaha SZ [2013-2...|        20000|2011| Individual|2nd owner|    21000|             NULL|
+--------------------+-------------+----+-----------+---------+---------+-----------------+
only showing top 5 rows


DataFrame[name: string, selling_price: int, year: int, seller_type: string, owner: string, km_driven: int, ex_showroom_price: int]

In [9]:
#import functions
from pyspark.ml.feature import SQLTransformer, VectorAssembler
from pyspark.ml import Pipeline
from pyspark.sql.types import StructType

In [10]:
#transform data with a SQLTransformer statement
sqlTrans = SQLTransformer(
    statement = """
                SELECT log(selling_price) AS label, year, log(km_driven) AS log_km_driven,
                       CASE WHEN owner = '1st owner' THEN 1 ELSE 0 END AS one_owner
                FROM __THIS__
                """
)
sqlTrans

SQLTransformer_3a95be93ba36

In [11]:
#assemble the modeling dataset with VectorAssembler statement
assembler = VectorAssembler(inputCols = ["year", "log_km_driven", "one_owner"], 
                            outputCol = "features", handleInvalid = "keep")
assembler

VectorAssembler_4c1c786810c0

In [12]:
#take a look
print(assembler.transform(sqlTrans.transform(fitDetails)))
assembler.transform(sqlTrans.transform(fitDetails)).show(5)

DataFrame[label: double, year: int, log_km_driven: double, one_owner: int, features: vector]
+------------------+----+------------------+---------+--------------------+
|             label|year|     log_km_driven|one_owner|            features|
+------------------+----+------------------+---------+--------------------+
|12.072541252905651|2019| 5.857933154483459|        1|[2019.0,5.8579331...|
|10.714417768752456|2017| 8.639410824140487|        1|[2017.0,8.6394108...|
|11.918390573078392|2018| 9.392661928770137|        1|[2018.0,9.3926619...|
|11.082142548877775|2015|10.043249494911286|        1|[2015.0,10.043249...|
| 9.903487552536127|2011|  9.95227771670556|        0|[2011.0,9.9522777...|
+------------------+----+------------------+---------+--------------------+
only showing top 5 rows


In [13]:
#make pipeline
pipeline = Pipeline(stages = [sqlTrans, assembler])

In [14]:
#fit this pipeline to the SQL data frame and save this as an object
pipeline_fit = pipeline.fit(fitDetails)
print(pipeline_fit)
pipeline_fit.params

PipelineModel_fa76d464a5b9


[]

In [15]:
#get schema to use  for streaming
fitDetails.schema

StructType([StructField('name', StringType(), True), StructField('selling_price', IntegerType(), True), StructField('year', IntegerType(), True), StructField('seller_type', StringType(), True), StructField('owner', StringType(), True), StructField('km_driven', IntegerType(), True), StructField('ex_showroom_price', IntegerType(), True)])

In [16]:
#set up schema for streaming
my_schema = StructType().add("name", "string") \
    .add("selling_price", "integer") \
    .add("year", "integer") \
    .add("seller_type", "string") \
    .add("owner", "string") \
    .add("km_driven", "integer") \
    .add("ex_showroom_price", "integer")
my_schema

StructType([StructField('name', StringType(), True), StructField('selling_price', IntegerType(), True), StructField('year', IntegerType(), True), StructField('seller_type', StringType(), True), StructField('owner', StringType(), True), StructField('km_driven', IntegerType(), True), StructField('ex_showroom_price', IntegerType(), True)])

In [17]:
#set up the streaming data frame
strDF = spark.readStream.option("header", "true").schema(my_schema).csv("csv_files")

In [18]:
#transfrom the streaming data frame using pipeline
trStrDF = pipeline_fit.transform(strDF)

In [19]:
query = trStrDF.writeStream.outputMode("append").format("console").start()

26/04/21 21:08:33 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-71e6de1d-3330-4afe-9c39-1361bfcb65e1. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/21 21:08:33 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


-------------------------------------------
Batch: 0
-------------------------------------------
+------------------+----+------------------+---------+--------------------+
|             label|year|     log_km_driven|one_owner|            features|
+------------------+----+------------------+---------+--------------------+
| 8.987196820661973|2003|10.887436932884098|        1|[2003.0,10.887436...|
|11.156250521031495|2018| 9.615805480084347|        1|[2018.0,9.6158054...|
|10.819778284410283|2016| 8.987196820661973|        1|[2016.0,8.9871968...|
| 10.46310334047155|2015|10.582738627903963|        1|[2015.0,10.582738...|
| 9.903487552536127|2006|11.225243392518447|        1|[2006.0,11.225243...|
|10.819778284410283|2012|10.239959789157341|        1|[2012.0,10.239959...|
| 10.51867319162636|2008| 11.03488966402723|        1|[2008.0,11.034889...|
|11.141861783579396|2018| 9.392661928770137|        1|[2018.0,9.3926619...|
|10.239959789157341|2012| 10.81975828421028|        1|[2012.0,10.81

In [20]:
#stop query
query.stop()

26/04/21 21:09:09 WARN DAGScheduler: Failed to cancel job group f589ee0f-4535-4aa8-b729-f128fc3c4ad6. Cannot find active jobs for it.
26/04/21 21:09:09 WARN DAGScheduler: Failed to cancel job group f589ee0f-4535-4aa8-b729-f128fc3c4ad6. Cannot find active jobs for it.
